<a href="https://colab.research.google.com/github/ashwinrajees-netizen/LLM-Research-Project/blob/main/GeneratingSubQuestions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U -q google-genai

In [ ]:
import json
import time
import os
from concurrent.futures import ThreadPoolExecutor
from google import genai
from tqdm.auto import tqdm

# --- 1. CONFIGURATION ---
API_KEY = "YOUR API KEY"
MODEL_ID = "gemini-3-flash-preview"
input_file = 'dataset.jsonl'
output_file = 'dataset_<Gemini3.1Flash>.jsonl'

# Adjust this based on your speed.
# Tier 1 can usually handle 10-15 workers comfortably.
MAX_WORKERS = 8

client = genai.Client(api_key=API_KEY)

SYSTEM_PROMPT = """Act as an Operational Logic Decomposition Engine.
Decompose the question into a numbered list of sub-questions.
Rules: 1. Single operation per step (+, -, *, /). 2. No data extraction. 3. Work-Unit Rule. 4. Referential Integrity."""

# --- 2. RESUME LOGIC ---
# This checks which rows you've already finished so you don't pay twice.
processed_indices = set()
if os.path.exists(output_file):
    with open(output_file, 'r') as f:
        for line in f:
            try:
                processed_indices.add(json.loads(line).get('index'))
            except:
                continue

# --- 3. LOAD REMAINING DATA ---
tasks = []
if not os.path.exists(input_file):
    print(f"Error: {input_file} not found in sidebar.")
else:
    with open(input_file, 'r') as f:
        for line in f:
            item = json.loads(line)
            if item.get('index') not in processed_indices:
                tasks.append(item)

# --- 4. THE WORKER FUNCTION ---
def process_question(data):
    # This function runs for each question in parallel
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=f"{SYSTEM_PROMPT}\n\nQuestion: {data['question']}"
        )
        data['decomposition'] = response.text
        return data
    except Exception as e:
        # If one fails, we mark it so you can see why later
        data['decomposition'] = f"ERROR: {str(e)}"
        return data

# --- 5. EXECUTION ---
if tasks:
    print(f"Resuming from {len(processed_indices)}. Processing remaining {len(tasks)} tasks...")

    with open(output_file, 'a') as outfile: # 'a' appends to the file
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # tqdm creates the progress bar
            for result in tqdm(executor.map(process_question, tasks), total=len(tasks), desc="Total Progress"):
                outfile.write(json.dumps(result) + '\n')
    print(f"\nBatch complete! Results saved in {output_file}")
else:
    print("All tasks already appear to be completed!")

In [ ]:
import json

# Replace with your actual file name
check_file = 'dataset_<Gemini3.1Flash>.jsonl'

with open(check_file, 'r') as f:
    lines = f.readlines()
    print(f"Total Lines in Output: {len(lines)}")

    # Check the last 3 lines for content
    print("\n--- Last 3 Results ---")
    for line in lines[-3:]:
        data = json.loads(line)
        print(f"Index {data.get('index')}: {data.get('decomposition')[:100]}...")

In [ ]:
!pip install anthropic tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os, time, anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

# --- CONFIG ---
# Anthropic API Key. Like the Gemini key, this should be stored securely.
ANTHROPIC_API_KEY = "YOUR API KEY"
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Specifies a Google Drive folder for output. This is crucial for persistent storage in Colab.
DRIVE_FOLDER = '/content/drive/MyDrive/Claude_Project'
INPUT_FILE = 'dataset.jsonl'
OUTPUT_FILE = f'{DRIVE_FOLDER}/dataset_Claude4.6.jsonl'
BATCH_SIZE = 700

# Helper function to get indices of already processed items from the Claude output file.
# This ensures the batch processing can resume correctly even if interrupted.
def get_processed_indices():
    processed = set()
    if os.path.exists(OUTPUT_FILE):
        # Opens and closes quickly to ensure the latest version from Drive is read.
        with open(OUTPUT_FILE, 'r') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    processed.add(int(data.get('index')))
                except: continue # Skips malformed lines.
    return processed

# The main loop for processing data using the Claude API in batches.
def run_master_loop():
    # Check if input file exists first
    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file '{INPUT_FILE}' not found. Please upload it to your Colab environment.")
        return

    # Determine total_rows dynamically
    with open(INPUT_FILE, 'r') as f:
        total_rows = sum(1 for line in f)

    while True:
        # 1. Update progress
        processed = get_processed_indices()
        remaining = total_rows - len(processed)

        print(f"\n{'='*40}")
        print(f"📊 CURRENT PROGRESS: {len(processed)} / {total_rows}")
        print(f"{'='*40}")

        if remaining <= 0:
            print("🎉 MISSION COMPLETE!")
            break # Exits the loop if all tasks are completed.

        # 2. Build Batch
        # Collects a batch of unprocessed questions to send to Claude.
        requests = []
        with open(INPUT_FILE, 'r') as f:
            for line in f:
                if len(requests) >= BATCH_SIZE: break # Limits batch size.
                data = json.loads(line)
                idx = int(data.get('index'))
                if idx not in processed:
                    requests.append(Request(
                        custom_id=f"idx_{idx}",
                        params=MessageCreateParamsNonStreaming(
                            model="claude-sonnet-4-6",
                            max_tokens=800,
                            # System prompt for Claude. 'cache_control' and 'thinking' are Claude-specific features.
                            system=[{"type": "text", "text": "Act as an Operational Logic Engine...", "cache_control": {"type": "ephemeral"}}],
                            messages=[{"role": "user", "content": data['question']}],
                            thinking={"type": "adaptive", "effort": "low"}
                        )
                    ))

        if not requests:
            print("No more unique rows found. Job finished?")
            break # Break if no new requests are found.

        # 3. Submit & Wait
        print(f"Submitting Batch for {len(requests)} rows (Batch ID pending)...")
        batch = client.messages.batches.create(requests=requests) # Submits the batch.

        while True:
            time.sleep(60) # Waits for a minute before checking batch status.
            status_obj = client.messages.batches.retrieve(batch.id) # Retrieves batch status.
            if status_obj.processing_status == "ended":
                print(f"\n✅ Batch {batch.id} finished.")
                break
            print(f"  > Claude processing... {status_obj.request_counts.succeeded} done", end='\r') # Live progress update.

        # 4. Save with FORCE FLUSH
        # Writes results from the completed batch to the output file on Google Drive.
        print("Writing results to Drive and syncing...")
        with open(OUTPUT_FILE, 'a') as outfile:
            for result in client.messages.batches.results(batch.id):
                clean_id = result.custom_id.replace('idx_', '').replace('idx-', '')
                if result.result.type == "succeeded":
                    res_text = result.result.message.content[0].text
                    outfile.write(json.dumps({"index": int(clean_id), "decomposition": res_text}) + '\n')

            # This forces Google Drive to "see" the new data. CRITICAL for incremental saves to Drive.
            outfile.flush()
            os.fsync(outfile.fileno())

        print("Done! Waiting 60 seconds for Drive to sync before next batch...")
        time.sleep(60) # CRITICAL: Gives Drive time to update its file size/content before the next iteration.

run_master_loop()

## Project Analysis and Annotation

This project is designed to process a dataset of 'questions' by decomposing them into sub-questions using two different Large Language Models (LLMs): Google's Gemini and Anthropic's Claude. The setup includes robust features for resuming interrupted processes, parallel execution for efficiency, and saving results to a file, including Google Drive for the Claude-processed outputs.

### 1. Gemini API Setup and Execution

This section handles the installation of the `google-genai` library and then sets up and executes the decomposition task using a Gemini model.

In [ ]:
import json
import time
import os
from concurrent.futures import ThreadPoolExecutor
from google import genai
from tqdm.auto import tqdm

# --- 1. CONFIGURATION ---
# The API_KEY needs to be replaced with your actual Google API Key.
# It's recommended to store API keys securely, e.g., using Colab secrets or environment variables.
API_KEY = "YOUR API KEY"

# The specific Gemini model ID to be used for the decomposition task.
MODEL_ID = "gemini-3-flash-preview"

# Input file containing the questions to be processed. Assumed to be in JSONL format.
input_file = 'dataset.jsonl'

# Output file where the Gemini-processed results will be stored, also in JSONL format.
output_file = 'dataset_<Gemini3.1Flash>.jsonl'

# Maximum number of worker threads to use for parallel processing of questions.
# This should be adjusted based on your API quota limits and desired throughput.
MAX_WORKERS = 8

# Initialize the Google GenAI client with the provided API key.
client = genai.Client(api_key=API_KEY)

# The system prompt given to the Gemini model to guide its response.
# It instructs the model to act as an 'Operational Logic Decomposition Engine' and decompose questions.
SYSTEM_PROMPT = """Act as an Operational Logic Decomposition Engine.\nDecompose the question into a numbered list of sub-questions.\nRules: 1. Single operation per step (+, -, *, /). 2. No data extraction. 3. Work-Unit Rule. 4. Referential Integrity."""

# --- 2. RESUME LOGIC ---
# This block implements a resume mechanism.
# It checks the output file for already processed indices to avoid re-processing and incurring duplicate costs.
processed_indices = set()
if os.path.exists(output_file):
    with open(output_file, 'r') as f:
        for line in f:
            try:
                processed_indices.add(json.loads(line).get('index'))
            except:
                # Skips lines that are not valid JSON or don't have an 'index'.
                continue

# --- 3. LOAD REMAINING DATA ---
# Loads the input data and filters out any questions that have already been processed.
tasks = []
if not os.path.exists(input_file):
    print(f"Error: {input_file} not found in sidebar.")
else:
    with open(input_file, 'r') as f:
        for line in f:
            item = json.loads(line)
            if item.get('index') not in processed_indices:
                tasks.append(item)

# --- 4. THE WORKER FUNCTION ---
# This function defines how each individual question is processed by the Gemini API.
# It sends the system prompt and the question to the model and stores the decomposition.
def process_question(data):
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=f"{SYSTEM_PROMPT}\n\nQuestion: {data['question']}"
        )
        data['decomposition'] = response.text
        return data
    except Exception as e:
        # In case of an API error, it records the error message in the 'decomposition' field.
        data['decomposition'] = f"ERROR: {str(e)}"
        return data

# --- 5. EXECUTION ---
# This is the main execution block for the Gemini processing.
# It uses a ThreadPoolExecutor to run the `process_question` function in parallel for all remaining tasks.
# `tqdm` provides a progress bar for monitoring.
# Results are appended to the output file incrementally.
if tasks:
    print(f"Resuming from {len(processed_indices)}. Processing remaining {len(tasks)} tasks...")

    with open(output_file, 'a') as outfile: # 'a' appends to the file
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # tqdm creates the progress bar
            for result in tqdm(executor.map(process_question, tasks), total=len(tasks), desc="Total Progress"):
                outfile.write(json.dumps(result) + '\n')
    print(f"\nBatch complete! Results saved in {output_file}")
else:
    print("All tasks already appear to be completed!")

### 2. Checking Gemini Output

This cell provides a quick way to inspect the generated output file from the Gemini processing, showing the total number of lines and a sample of the last few results.

In [ ]:
import json

# Specifies the output file to check.
check_file = 'dataset_<Gemini3.1Flash>.jsonl'

# Reads all lines from the file.
with open(check_file, 'r') as f:
    lines = f.readlines()
    print(f"Total Lines in Output: {len(lines)}")

    # Prints the last 3 lines (results) to quickly verify the content and format.
    print("\n--- Last 3 Results ---")
    for line in lines[-3:]:
        data = json.loads(line)
        # Prints the index and the first 100 characters of the 'decomposition'.
        print(f"Index {data.get('index')}: {data.get('decomposition')[:100]}...")

### 3. Claude API Setup and Execution

This section sets up the environment for using Anthropic's Claude API, including installing the necessary library and configuring Google Drive for persistent storage of results. It then defines a sophisticated batch processing loop to handle large datasets efficiently.

In [ ]:
!pip install anthropic tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os, time, anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

# --- CONFIG ---
# Anthropic API Key. Like the Gemini key, this should be stored securely.
ANTHROPIC_API_KEY = "YOUR API KEY"
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Specifies a Google Drive folder for output. This is crucial for persistent storage in Colab.
DRIVE_FOLDER = '/content/drive/MyDrive/Claude_Project'
INPUT_FILE = 'dataset.jsonl'
OUTPUT_FILE = f'{DRIVE_FOLDER}/dataset_Claude4.6.jsonl'
BATCH_SIZE = 700

# Helper function to get indices of already processed items from the Claude output file.
# This ensures the batch processing can resume correctly even if interrupted.
def get_processed_indices():
    processed = set()
    if os.path.exists(OUTPUT_FILE):
        # Opens and closes quickly to ensure the latest version from Drive is read.
        with open(OUTPUT_FILE, 'r') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    processed.add(int(data.get('index')))
                except: continue # Skips malformed lines.
    return processed

# The main loop for processing data using the Claude API in batches.
def run_master_loop():
    while True:
        # 1. Update progress
        processed = get_processed_indices()
        total_rows = 8792 # Hardcoded total number of rows.
        remaining = total_rows - len(processed)

        print(f"\n{'='*40}")
        print(f"📊 CURRENT PROGRESS: {len(processed)} / {total_rows}")
        print(f"{'='*40}")

        if remaining <= 0:
            print("🎉 MISSION COMPLETE!")
            break # Exits the loop if all tasks are completed.

        # 2. Build Batch
        # Collects a batch of unprocessed questions to send to Claude.
        requests = []
        with open(INPUT_FILE, 'r') as f:
            for line in f:
                if len(requests) >= BATCH_SIZE: break # Limits batch size.
                data = json.loads(line)
                idx = int(data.get('index'))
                if idx not in processed:
                    requests.append(Request(
                        custom_id=f"idx_{idx}",
                        params=MessageCreateParamsNonStreaming(
                            model="claude-sonnet-4-6",
                            max_tokens=800,
                            # System prompt for Claude. 'cache_control' and 'thinking' are Claude-specific features.
                            system=[{"type": "text", "text": "Act as an Operational Logic Engine...", "cache_control": {"type": "ephemeral"}}],
                            messages=[{"role": "user", "content": data['question']}],
                            thinking={"type": "adaptive", "effort": "low"}
                        )
                    ))

        if not requests:
            print("No more unique rows found. Job finished?")
            break # Break if no new requests are found.

        # 3. Submit & Wait
        print(f"Submitting Batch for {len(requests)} rows (Batch ID pending)...")
        batch = client.messages.batches.create(requests=requests) # Submits the batch.

        while True:
            time.sleep(60) # Waits for a minute before checking batch status.
            status_obj = client.messages.batches.retrieve(batch.id) # Retrieves batch status.
            if status_obj.processing_status == "ended":
                print(f"\n✅ Batch {batch.id} finished.")
                break
            print(f"  > Claude processing... {status_obj.request_counts.succeeded} done", end='\r') # Live progress update.

        # 4. Save with FORCE FLUSH
        # Writes results from the completed batch to the output file on Google Drive.
        print("Writing results to Drive and syncing...")
        with open(OUTPUT_FILE, 'a') as outfile:
            for result in client.messages.batches.results(batch.id):
                clean_id = result.custom_id.replace('idx_', '').replace('idx-', '')
                if result.result.type == "succeeded":
                    res_text = result.result.message.content[0].text
                    outfile.write(json.dumps({"index": int(clean_id), "decomposition": res_text}) + '\n')

            # This forces Google Drive to "see" the new data. CRITICAL for incremental saves to Drive.
            outfile.flush()
            os.fsync(outfile.fileno())

        print("Done! Waiting 60 seconds for Drive to sync before next batch...")
        time.sleep(60) # CRITICAL: Gives Drive time to update its file size/content before the next iteration.

run_master_loop()

### Project Summary and Impact

This project establishes a high-throughput, resilient pipeline for decomposing complex operational logic into atomic sub-questions. By leveraging state-of-the-art LLMs from both Google and Anthropic, the system provides a dual-model approach to logic verification and data processing.

#### Summary of Technical Achievements
*   **Hybrid API Strategy**: Successfully integrates the **Google Gemini SDK** for real-time parallel processing and the **Anthropic Batch API** for high-volume asynchronous execution.
*   **Resilience and Persistence**: Features a custom-built **Resume Logic** that tracks indices to prevent data loss or duplicate API costs. Integration with **Google Drive** ensures that large-scale batch results are persisted outside the ephemeral Colab environment.
*   **Operational Integrity**: Implements strict 'Referential Integrity' and 'Work-Unit' rules via system prompting, ensuring the resulting sub-questions are logically sound and mathematically decomposable.

#### Strategic Impact
*   **Scalability**: The architecture is designed to scale from a few dozen questions to tens of thousands (as seen with the 8,792 task target) without manual intervention.
*   **Cost Efficiency**: By using resume logic and efficient batch processing, the project minimizes API overhead and prevents re-processing fees during network or runtime interruptions.
*   **Data Quality**: The use of structured JSONL output and forced file synchronization (`os.fsync`) ensures that the resulting dataset is ready for immediate use in downstream machine learning tasks or logic-based applications.